# Actividad Extra: Analisis de Ventas por Tienda

## Objetivos
- Leer y combinar multiples datasets con join
- Analizar ventas totales por tienda y departamento
- Explorar la evolucion temporal de las ventas
- Crear visualizaciones con matplotlib
- Analizar la relacion entre tamano de tienda y ventas

## Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import matplotlib.pyplot as plt
import pandas as pd

spark = SparkSession.builder \
    .appName("actividad_ventas") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

---
## Ejercicio 1: Lectura y join de datasets

Lee los archivos `sales.csv` y `stores.csv`, realiza un join por la columna `Store` y explora el resultado.

**Columnas sales.csv:** Store, Dept, Date, Weekly_Sales, IsHoliday

**Columnas stores.csv:** Store, Type, Size

In [ ]:
# =============================================================
# EJERCICIO 1: Lectura y join de datasets
# =============================================================

# Leer ambos archivos
df_sales = spark.read.csv("/home/jovyan/datos/sales.csv", header=True, inferSchema=True)
df_stores = spark.read.csv("/home/jovyan/datos/stores.csv", header=True, inferSchema=True)

# Explorar cada uno
print("=== Schema sales ===")
df_sales.printSchema()
print(f"Registros sales: {df_sales.count()}")

print("\n=== Schema stores ===")
df_stores.printSchema()
print(f"Registros stores: {df_stores.count()}")

# Join por Store
df = df_sales.join(df_stores, on="Store", how="inner")

print(f"\n=== Resultado del join ===")
df.printSchema()
df.show(5)
print(f"Total registros: {df.count()}")

> **Nota docente:** El join es una operacion fundamental en analisis de datos. Al usar `on="Store"`, Spark elimina automaticamente la columna duplicada del segundo DataFrame. Si se usara la sintaxis `df_sales.join(df_stores, df_sales.Store == df_stores.Store)`, la columna Store apareceria duplicada y habria que eliminar una manualmente. El `how="inner"` es el tipo por defecto y solo mantiene filas que existen en ambas tablas. Los alumnos deben verificar que el conteo despues del join sea coherente (no se pierdan ni dupliquen registros inesperadamente).

---
## Ejercicio 2: Ventas totales por tienda

Calcula las ventas totales (`Weekly_Sales`) por cada tienda, ordena de mayor a menor y muestra el top 10.

In [ ]:
# =============================================================
# EJERCICIO 2: Ventas totales por tienda
# =============================================================

# Agrupar por tienda y sumar ventas
df_ventas_tienda = df.groupBy("Store").agg(
    F.sum("Weekly_Sales").alias("total_ventas")
).orderBy("total_ventas", ascending=False)

# Mostrar top 10
print("Top 10 tiendas por ventas totales:")
df_ventas_tienda.show(10)

> **Nota docente:** La combinacion `groupBy().agg()` es mas flexible que `groupBy().sum()` porque permite renombrar la columna resultante con `.alias()`. Sin alias, la columna se llamaria `sum(Weekly_Sales)` que es mas dificil de referenciar en operaciones posteriores. Los alumnos deben notar que `Weekly_Sales` puede contener valores negativos (devoluciones), por lo que la suma total representa ventas netas.

---
## Ejercicio 3: Ventas por tipo de tienda

Analiza como varian las ventas segun el tipo de tienda (columna `Type`: A, B, C). Calcula ventas totales y promedio por tipo.

In [ ]:
# =============================================================
# EJERCICIO 3: Ventas por tipo de tienda
# =============================================================

# Agrupar por tipo y calcular metricas
df_por_tipo = df.groupBy("Type").agg(
    F.sum("Weekly_Sales").alias("total_ventas"),
    F.avg("Weekly_Sales").alias("promedio_ventas"),
    F.count("*").alias("num_registros")
).orderBy("total_ventas", ascending=False)

df_por_tipo.show()

> **Nota docente:** El uso de multiples funciones de agregacion dentro de un solo `.agg()` es un patron eficiente porque Spark calcula todas las metricas en una sola pasada sobre los datos. Los tipos de tienda (A, B, C) corresponden a diferentes tamanos y formatos de tienda. Es interesante discutir con los alumnos que aunque el tipo A puede tener mas ventas totales, no necesariamente tiene el mayor promedio por registro.

---
## Ejercicio 4: Evolucion mensual de ventas

Parsea la columna `Date`, extrae el anio y mes, calcula ventas totales por mes y crea un grafico de linea temporal.

In [ ]:
# =============================================================
# EJERCICIO 4: Evolucion mensual de ventas
# =============================================================

# Parsear la fecha y extraer anio-mes
df_temporal = df.withColumn("fecha_parsed", F.to_date("Date", "yyyy-MM-dd"))

# Si el formato anterior no funciona, probar otros formatos comunes
# df_temporal = df.withColumn("fecha_parsed", F.to_date("Date", "dd/MM/yyyy"))
# df_temporal = df.withColumn("fecha_parsed", F.to_date("Date", "MM/dd/yyyy"))

df_temporal = df_temporal.withColumn("anio_mes",
    F.date_format("fecha_parsed", "yyyy-MM")
)

# Agrupar por anio-mes y sumar ventas
df_mensual = df_temporal.groupBy("anio_mes").agg(
    F.sum("Weekly_Sales").alias("total_ventas")
).orderBy("anio_mes")

# Convertir a Pandas y graficar
pdf_mensual = df_mensual.toPandas()

plt.figure(figsize=(14, 5))
plt.plot(pdf_mensual["anio_mes"], pdf_mensual["total_ventas"], marker="o", linewidth=1.5)
plt.title("Evolucion Mensual de Ventas Totales", fontsize=14)
plt.xlabel("Mes")
plt.ylabel("Ventas Totales ($)")
plt.xticks(rotation=45, ha="right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

> **Nota docente:** El parseo de fechas puede ser problematico si el formato no coincide. Es buena practica verificar una muestra de la columna `Date` antes de elegir el formato. `F.date_format()` permite formatear una fecha a string con el patron deseado. Al convertir a Pandas con `.toPandas()`, todos los datos se mueven al driver, lo cual es viable para datos agregados pero no para el DataFrame completo si es muy grande. Error comun: los alumnos pueden olvidar `plt.tight_layout()` y las etiquetas se solapan.

---
## Ejercicio 5: Tamano de tienda vs ventas (scatter plot)

Calcula las ventas totales por tienda, une con el tamano de la tienda y crea un scatter plot de Size vs total de ventas.

In [ ]:
# =============================================================
# EJERCICIO 5: Tamano de tienda vs ventas
# =============================================================

# Ventas totales por tienda con info de la tienda
df_tienda_size = df.groupBy("Store", "Size", "Type").agg(
    F.sum("Weekly_Sales").alias("total_ventas")
)

# Convertir a Pandas
pdf_scatter = df_tienda_size.toPandas()

# Scatter plot coloreado por tipo
colores = {"A": "#2196F3", "B": "#FF9800", "C": "#4CAF50"}

plt.figure(figsize=(10, 6))
for tipo in sorted(pdf_scatter["Type"].unique()):
    mask = pdf_scatter["Type"] == tipo
    plt.scatter(
        pdf_scatter.loc[mask, "Size"],
        pdf_scatter.loc[mask, "total_ventas"],
        label=f"Tipo {tipo}",
        c=colores.get(tipo, "gray"),
        alpha=0.7,
        s=80
    )

plt.title("Tamano de Tienda vs Ventas Totales", fontsize=14)
plt.xlabel("Tamano de Tienda (sq ft)")
plt.ylabel("Ventas Totales ($)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

> **Nota docente:** El scatter plot revela la correlacion entre tamano de tienda y ventas. Al agrupar por Store, Size y Type en el mismo `groupBy`, evitamos hacer un join adicional. La coloracion por tipo de tienda agrega una tercera dimension al analisis. Los alumnos pueden observar que existe una correlacion positiva entre tamano y ventas, y que los tipos de tienda tienden a agruparse en rangos de tamano especificos. Para un analisis mas formal, se podria calcular el coeficiente de correlacion de Pearson.

---
## Ejercicio 6: Top 10 departamentos por ventas

Encuentra los 10 departamentos (`Dept`) con mayores ventas totales considerando todas las tiendas.

In [ ]:
# =============================================================
# EJERCICIO 6: Top 10 departamentos por ventas
# =============================================================

# Agrupar por departamento y sumar ventas
df_dept = df.groupBy("Dept").agg(
    F.sum("Weekly_Sales").alias("total_ventas")
).orderBy("total_ventas", ascending=False).limit(10)

# Mostrar tabla
df_dept.show()

# Grafico de barras horizontales
pdf_dept = df_dept.toPandas()

plt.figure(figsize=(10, 5))
plt.barh(
    pdf_dept["Dept"].astype(str),
    pdf_dept["total_ventas"],
    color="#2196F3"
)
plt.title("Top 10 Departamentos por Ventas Totales", fontsize=14)
plt.xlabel("Ventas Totales ($)")
plt.ylabel("Departamento")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

> **Nota docente:** El metodo `limit(10)` toma las primeras N filas del DataFrame ya ordenado, equivalente a `LIMIT 10` en SQL. Es importante que `orderBy()` se ejecute antes de `limit()` para obtener el top correcto. `gca().invert_yaxis()` invierte el eje Y para que el departamento con mas ventas aparezca arriba. El conversion a string del Dept es necesario para que matplotlib lo trate como categorico y no como numerico. Los alumnos pueden investigar que representa cada numero de departamento en el contexto de Walmart (origen de este dataset).

---
## Ejercicio 7: Impacto de los festivos en las ventas

Analiza si las semanas festivas (`IsHoliday = true`) tienen un impacto significativo en las ventas comparado con semanas normales.

In [ ]:
# =============================================================
# EJERCICIO 7: Impacto de los festivos
# =============================================================

# Agrupar por IsHoliday y calcular metricas
df_festivos = df.groupBy("IsHoliday").agg(
    F.avg("Weekly_Sales").alias("promedio_ventas"),
    F.sum("Weekly_Sales").alias("total_ventas"),
    F.count("*").alias("num_registros")
)

df_festivos.show()

# Grafico de barras comparativo
pdf_festivos = df_festivos.toPandas()
pdf_festivos["etiqueta"] = pdf_festivos["IsHoliday"].map({True: "Festivo", False: "No Festivo"})

plt.figure(figsize=(8, 5))
plt.bar(
    pdf_festivos["etiqueta"],
    pdf_festivos["promedio_ventas"],
    color=["#FF5722", "#607D8B"]
)
plt.title("Ventas Promedio: Festivo vs No Festivo", fontsize=14)
plt.ylabel("Ventas Promedio Semanales ($)")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

> **Nota docente:** Este ejercicio introduce el analisis comparativo entre grupos. Los festivos en este dataset incluyen Super Bowl, Dia del Trabajo, Accion de Gracias y Navidad. Es esperable que las semanas festivas tengan un promedio de ventas superior. Punto de discusion: aunque las semanas festivas tienen mayor promedio, hay muchas menos semanas festivas que normales, por lo que el total de ventas es menor. Los alumnos pueden profundizar analizando que festivos especificos generan mas ventas con un filtro adicional por fecha.

---
## Resumen

En esta actividad aprendimos:

1. **Join de datasets** con `.join()` para combinar informacion de multiples fuentes
2. **Agregaciones** con `groupBy().agg()` para calcular metricas de negocio
3. **Analisis temporal** parseando fechas y agrupando por periodos
4. **Visualizaciones** convirtiendo a Pandas y usando matplotlib
5. **Scatter plots** para analizar relaciones entre variables
6. **Rankings** con `orderBy()` y `limit()` para top N
7. **Comparaciones** entre grupos (festivo vs no festivo)

In [ ]:
# Detener SparkSession
spark.stop()
print("SparkSession detenida correctamente.")